# Carnelian Compass: how to trust an impact score
Data-quality and validation framework for nonprofit and ESG scoring, on public data.

*Independent case study based on public data · not affiliated with or endorsed by Carnelian Ventures.* Real records are labelled **PUBLIC DATA**; injected errors are **SIMULATED**. The demo score is illustrative, not Carnelian's methodology. This notebook reads the outputs of `python src/run_all.py`.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
OUT = ROOT / 'data' / 'outputs'
from prepare import load_clean
pd.set_option('display.width', 140)
t = load_clean()
{k: v.shape for k, v in t.items()}

{'us': (2032, 55),
 'uk': (500, 29),
 'companies': (300, 32),
 'power': (1071, 17)}

## 1. Data profile (PUBLIC DATA)
Records per source, and the share of missing values in the fields the score needs.

In [2]:
us = t['us']
key = ['revenue', 'expenses', 'assets', 'exp_program', 'exp_fundraising', 'contributions', 'revenue_py', 'ntee_cd', 'audited', 'soi_revenue']
us[key].isna().mean().round(3).rename('share missing').to_frame().T

,revenue,expenses,assets,exp_program,exp_fundraising,contributions,revenue_py,ntee_cd,audited,soi_revenue
share missing,0.0,0.0,0.0,0.0,0.0,0.0,0.03,0.133,0.002,0.002


In [3]:
us.drop_duplicates('ein').groupby('size_band').agg(orgs=('ein', 'nunique'), median_revenue=('revenue', 'median'),
    ntee_missing=('ntee_cd', lambda s: s.isna().mean()), audited=('audited', lambda s: s.astype(float).mean())).round(3)

,orgs,median_revenue,ntee_missing,audited
size_band,,,,
L $5-50m,500,11112600.5,0.148,0.622
M $0.5-5m,500,1281123.5,0.168,0.403
S <$0.5m,500,212694.0,0.132,0.104
XL >$50m,500,107196257.0,0.088,0.365


## 2. Rule results
40 rules over six dimensions, run in pandas **and** DuckDB SQL (the pipeline stops if they disagree).

In [4]:
rr = pd.read_csv(OUT / 'rule_results.csv')
rr.groupby(['dimension', 'severity']).agg(rules=('id', 'size'), flags_real=('flagged_real', 'sum')).unstack(fill_value=0)

rules              flags_real              
severity     error info warning      error  info warning
dimension                                               
accuracy         3    1       4         53     0     700
completeness     2    1       3          0   190     301
consistency      3    0       6         18     0      84
freshness        2    1       1         26  2000      41
uniqueness       2    1       1         63     4       0
validity         6    0       3         11     0      15

In [5]:
rr[rr.flagged_real > 0].sort_values('flagged_real', ascending=False)[['id', 'severity', 'description', 'flagged_real', 'pass_rate_real']].head(12)

,id,severity,description,flagged_real,pass_rate_real
29,FRESH-02,info,The IRS master file already lists a more recen...,2000,0.015748
38,ACC-07,warning,EPA parent company and Climate TRACE owner sha...,431,0.597572
2,COMP-03,warning,NTEE activity code is present (needed for peer...,271,0.866634
37,ACC-06,warning,Climate TRACE estimate is within 0.5x to 1.5x ...,221,0.793651
4,COMP-05,info,Revenue is available (needed for carbon intens...,190,0.366667
24,UNIQ-01,error,One return per EIN for the tax year.,63,0.968996
31,FRESH-04,warning,The charity is not overdue or in double defaul...,41,0.918000
36,ACC-05,warning,Register income = income in the latest annual ...,38,0.924000
18,CONS-04,warning,Revenue is within 0.2x to 5x of the prior year...,37,0.981791
19,CONS-05,warning,Expenses are within 0.1x to 10x of revenue.,25,0.987697


In [6]:
pd.read_csv(OUT / 'health.csv').pivot_table(index=['dataset', 'table'], columns='dimension', values='score').round(3)

dimension                                   accuracy  completeness  consistency  freshness  uniqueness  validity
dataset                          table                                                                          
real (PUBLIC DATA)               companies       NaN         0.977        0.970        NaN       1.000     1.000
                                 power         0.481           NaN          NaN        NaN         NaN     0.995
                                 uk            0.924         0.954        0.974      0.918       1.000     1.000
                                 us            0.988         0.867        0.965      0.999       0.969     0.990
with injected errors (SIMULATED) companies       NaN         0.977        0.903        NaN       1.000     1.000
                                 power         0.469           NaN          NaN        NaN         NaN     0.995
                                 uk            0.909         0.954        0.974      0.903       0.980     0.939
                                 us            0.933         0.854        0.930      0.989       0.945     0.954

## 3. Reconciliation between sources

In [7]:
pd.read_csv(OUT / 'reconciliation_summary.csv')[['source_pair', 'measure', 'compared', 'agree_rate', 'note']]

,source_pair,measure,compared,agree_rate,note
0,IRS e-file vs IRS SOI extract,revenue,1987,0.994967,same EIN and fiscal period; tolerance 1%
1,IRS e-file vs IRS SOI extract,expenses,1987,0.995471,same EIN and fiscal period; tolerance 1%
2,IRS e-file vs IRS SOI extract,assets,1987,0.994967,same EIN and fiscal period; tolerance 1%
3,IRS e-file vs IRS SOI extract,fiscal period found,2000,0.993500,SOI record for the same fiscal period
4,IRS e-file vs IRS BMF,revenue (same period only),30,0.966667,BMF shows the same period for 2% of orgs only
5,IRS e-file vs IRS BMF,BMF period is more recent,2000,0.984500,share of orgs where a newer return already exists
6,IRS e-file vs ProPublica,fiscal period found,160,0.475000,158 of 160 orgs returned filing data
7,IRS e-file vs ProPublica,revenue,76,0.973684,same EIN and fiscal period; tolerance 1%
8,IRS e-file vs ProPublica,expenses,76,0.973684,same EIN and fiscal period; tolerance 1%
9,IRS e-file vs ProPublica,assets,76,0.973684,same EIN and fiscal period; tolerance 1%


In [8]:
mc = pd.read_csv(OUT / 'manual_check.csv')
mc.groupby('field').match.mean().rename('match rate vs original return').to_frame().T

field,assets,exp_fundraising,exp_management,exp_program,exp_total_p9,expenses,liabilities,net_assets,revenue
match rate vs original return,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 4. Detection of the 300 injected errors (SIMULATED)
Recall = injected errors caught; precision = share of new alerts that point at an injected error; baseline = alerts the same rules raise on the untouched real data.

In [9]:
pd.read_csv(OUT / 'detection_by_type.csv')[['label', 'injected', 'detected', 'recall', 'precision', 'designed_rules', 'baseline_alerts_real_data']]

,label,injected,detected,recall,precision,designed_rules,baseline_alerts_real_data
0,Wrong parent company,30,18,0.600000,1.0,ACC-07,431
1,GBP treated as USD,30,26,0.866667,1.0,VAL-07,0
2,Negative amounts,30,29,0.966667,1.0,"VAL-01, VAL-02, VAL-06, VAL-08",11
3,Missing (small orgs),30,29,0.966667,1.0,COMP-02,0
4,Stale filing used,30,29,0.966667,1.0,"FRESH-01, FRESH-03",26
5,Duplicates,30,30,1.000000,1.0,"UNIQ-01, UNIQ-03",63
6,Fiscal year shifted,30,30,1.000000,1.0,"VAL-05, ACC-04",10
7,Outliers x10,30,30,1.000000,1.0,"CONS-04, ACC-01, CONS-08",64
8,Parts do not add up,30,30,1.000000,1.0,CONS-01,16
9,Units (thousands),30,30,1.000000,1.0,"ACC-01, ACC-02, ACC-03, CONS-09, CONS-08",62


In [10]:
d = pd.read_csv(OUT / 'detection_detail.csv')
d[~d.detected].groupby(['table', 'error_type']).size().rename('missed')

table  error_type     
power  parent_mismatch    12
uk     currency            4
       stale               1
us     missing_small       1
       signs               1
Name: missed, dtype: int64

## 5. Score sensitivity

In [11]:
s = json.loads((OUT / 'robustness_summary.json').read_text())
pd.DataFrame(s).round(3)

,nonprofit,company
scored,1995.000,293.000
insufficient,5.000,7.000
blocked_real,62.000,9.000
tau_median,0.788,0.894
tau_p05,0.643,0.788
mean_abs_rank_change_pts,8.018,5.343
share_within_10pts,0.683,0.852
top_decile_overlap,0.779,0.828
injected_records_scored,175.000,20.000
injected_median_rank_change_pts,8.627,36.177


In [12]:
pd.concat([pd.read_csv(OUT / f'robustness_oat_{k}.csv').assign(kind=k) for k in ('nonprofit', 'company')]).round(3)

,pillar,factor,tau,scores_lost,scores_gained,kind
0,financial_health,0.5,0.786,0,0,nonprofit
1,financial_health,1.5,0.845,0,0,nonprofit
2,efficiency,0.5,0.807,0,0,nonprofit
3,efficiency,1.5,0.848,0,0,nonprofit
4,stability,0.5,0.842,0,0,nonprofit
5,stability,1.5,0.869,0,0,nonprofit
6,transparency,0.5,0.864,0,0,nonprofit
7,transparency,1.5,0.885,0,0,nonprofit
0,carbon_intensity,0.5,0.914,0,0,company
1,carbon_intensity,1.5,0.901,183,0,company


## 6. Cases that need more research (anonymised)
Open research items by rule, and three anonymised examples where sources contradict each other.

In [13]:
log = pd.read_csv(OUT / 'issues_log.csv')
log[log.status == 'Open - research'].groupby(['rule_id', 'severity']).size().rename('open').sort_values(ascending=False).head(10)

rule_id   severity
ACC-07    warning     431
ACC-06    warning     221
FRESH-04  warning      41
ACC-05    warning      38
CONS-04   warning      37
CONS-05   warning      25
ACC-03    error        18
ACC-01    error        18
ACC-02    error        17
CONS-01   error        16
Name: open, dtype: int64

In [14]:
soi = pd.read_csv(OUT / 'recon_irs_soi.csv').sort_values('rev_gap', ascending=False).head(3)
soi.assign(case=[f'Nonprofit #{i + 1}' for i in range(len(soi))], ratio=(soi.revenue / soi.soi_revenue).round(3))[['case', 'size_band', 'ratio']]

,case,size_band,ratio
1981,Nonprofit #1,M $0.5-5m,1.907
1980,Nonprofit #2,M $0.5-5m,0.308
1845,Nonprofit #3,M $0.5-5m,1.111


**Reading**: every IRS-side disagreement sits on an amended return; the fix is to join on EIN + fiscal period and keep the latest return, while logging both. Model-vs-report gaps on power plants are logged, not averaged.